# Advanced Machine Learning for Speech Emotion Recognition (SER)

This notebook contains the complete pipeline for tuning, training, and evaluating advanced machine learning models (**XGBoost**, **LightGBM**, and **Support Vector Classifier (SVC)**) on pre-extracted acoustic features. 

The goal is to classify 6 distinct emotion classes: *anger, disgust, fear, happy, neutral, sad*.


In [1]:
# ============================================================
# Library Imports
# ============================================================
import os
import sys
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import optuna
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, f1_score, cohen_kappa_score
from sklearn.svm import SVC

try:
    import xgboost as xgb
except ImportError:
    # Auto-install xgboost if not present
    %pip install xgboost -q
    import xgboost as xgb

try:
    import lightgbm as lgb
except ImportError:
    # Auto-install lightgbm if not present
    %pip install lightgbm -q
    import lightgbm as lgb

warnings.filterwarnings('ignore')


c:\Users\User\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Configuration & Environmental Setup

We determine the project root directory and set path references for the audio feature dataset (`all_emotions.csv`). We also define constants such as `RANDOM_STATE` for reproducibility.


In [2]:
# ============================================================
# Path Configuration and Global Constants
# ============================================================
_base = os.getcwd()
_project_root = os.path.dirname(_base) if os.path.basename(_base).lower() == "training" else _base
ALL_EMOTIONS_CSV = os.path.normpath(os.path.join(_project_root, "dataset", "all_emotions.csv"))
if not os.path.isfile(ALL_EMOTIONS_CSV):
    fallback_csv = os.path.normpath(os.path.join(_project_root, "all_emotions.csv"))
    if os.path.isfile(fallback_csv):
        ALL_EMOTIONS_CSV = fallback_csv

print("Dataset path:", ALL_EMOTIONS_CSV)
print("Dataset file exists:", os.path.isfile(ALL_EMOTIONS_CSV))

RANDOM_STATE = 42
TEST_SIZE = 0.2


Dataset path: c:\Users\User\Downloads\Saggin-Computational-Architectures-in-Speech-Emotion-Recognition-main\Saggin-Computational-Architectures-in-Speech-Emotion-Recognition-main\all_emotions.csv
Dataset file exists: True


## Exploratory Data Analysis & Target Distribution

We load the tabular dataset and inspect the counts and distribution of emotion labels in the raw dataset before preprocessing.


In [3]:
# ============================================================
# Data Loading and Exploration
# ============================================================
df = pd.read_csv(ALL_EMOTIONS_CSV)
print("Raw Dataset Shape:", df.shape)

# Identify target column (label / Label)
TARGET_COL = "label"
if TARGET_COL not in df.columns and "Label" in df.columns:
    TARGET_COL = "Label"

print("\n--- Target Class Distribution ---")
print(df[TARGET_COL].astype(str).value_counts(dropna=False))


Raw Dataset Shape: (54485, 49)

--- Target Class Distribution ---
label
anger      9315
disgust    9315
fear       9315
happy      9315
sad        9310
neutral    7915
Name: count, dtype: int64


## Feature Selection & Imputation

To optimize training execution speed and prevent model overfitting, we train the models using a subset of **26 highly-informative features** selected based on Random Forest feature importances:
*   **Pitch/F0**: `F0_mean`, `F0_std`, `F0_range`
*   **Energy**: `Energy_ mean`, `Energy_ std`
*   **Zero Crossing Rate (ZCR)**: `ZCR_mean`, `ZCR_std`
*   **Spectral Descriptors**: `Spectral_centroid_mean`, `Spectral_centroid_std`, `Spectral_flux_mean`
*   **MFCC Means & Stds**: `MFCC_C0_mean`, `MFCC_C1_mean`, `MFCC_C2_mean`, `MFCC_C3_mean`, `MFCC_C5_mean`, `MFCC_C7_mean`, `MFCC_C10_mean`, `MFCC_C0_std`, `MFCC_C1_std`, `MFCC_C2_std`, `MFCC_C3_std`, `MFCC_C5_std`, `MFCC_C7_std`, `Delta_MFCC_C0_std`, `Delta_MFCC_C2_std`, `Delta_MFCC_C3_std`

We clean null labels and apply **median imputation** to handle any missing feature values.


In [4]:
# ============================================================
# Feature Selection and Data Cleaning
# ============================================================
# Pruned subset of 26 highly-informative features selected based on Random Forest feature importances
FEATURE_COLS = [
    "F0_mean", "F0_std", "F0_range", 
    "Energy_ mean", "Energy_ std", 
    "ZCR_mean", "ZCR_std", 
    "Spectral_centroid_mean", "Spectral_centroid_std", "Spectral_flux_mean", 
    "MFCC_C0_mean", "MFCC_C1_mean", "MFCC_C2_mean", "MFCC_C3_mean", 
    "MFCC_C5_mean", "MFCC_C7_mean", "MFCC_C10_mean", 
    "MFCC_C0_std", "MFCC_C1_std", "MFCC_C2_std", "MFCC_C3_std", 
    "MFCC_C5_std", "MFCC_C7_std", 
    "Delta_MFCC_C0_std", "Delta_MFCC_C2_std", "Delta_MFCC_C3_std"
]

# Drop rows with null/missing targets or 'nan' string labels
df_cleaned = df.dropna(subset=[TARGET_COL]).copy()
df_cleaned = df_cleaned[df_cleaned[TARGET_COL].astype(str).str.strip().str.lower() != "nan"]

# Feature Imputation: Replace NaN / Inf values in feature columns with column medians
for col in FEATURE_COLS:
    s = pd.to_numeric(df_cleaned[col], errors="coerce")
    s = s.replace([np.inf, -np.inf], np.nan)
    med = s.median()
    if pd.isna(med):
        med = 0.0
    df_cleaned[col] = s.fillna(med)

X = df_cleaned[FEATURE_COLS].copy()
y_label = df_cleaned[TARGET_COL].astype(str).str.strip()

print("Cleaned X shape:", X.shape)
print("Cleaned y label shape:", y_label.shape)


Cleaned X shape: (54485, 26)
Cleaned y label shape: (54485,)


## Data Splitting & Feature Scaling

We perform a stratified train-test split (80% training, 20% validation/testing) to preserve the label distribution. We then apply `StandardScaler` to normalize features (z-score normalization), which is especially critical for distance-based models like SVM.


In [5]:
# ============================================================
# Label Encoding, Train-Test Split, and Feature Scaling
# ============================================================
le = LabelEncoder()
y = le.fit_transform(y_label)

print("Label Mapping:")
for i, name in enumerate(le.classes_):
    print(f"  {i} -> {name}")

# Perform Stratified Train-Test Split (80% Train, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(
    X.values, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)

# Scale features using StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Train shape:", X_train_scaled.shape, "| Test shape:", X_test_scaled.shape)


Label Mapping:
  0 -> anger
  1 -> disgust
  2 -> fear
  3 -> happy
  4 -> neutral
  5 -> sad
Train shape: (43588, 26) | Test shape: (10897, 26)


## Hyperparameter Tuning with Optuna

We leverage **Optuna** to optimize hyperparameter settings for the three classifiers on the training dataset.

### 1. XGBoost Hyperparameter Search
We search over tree depth, learning rate, subsample ratio, colsample bytree, and gamma using a validation split.


In [6]:
# ============================================================
# XGBoost Hyperparameter Optimization (Optuna)
# ============================================================
def objective_xgb(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 500),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'gamma': trial.suggest_float('gamma', 1e-8, 1.0, log=True),
        'random_state': RANDOM_STATE,
        'n_jobs': -1,
        'eval_metric': 'mlogloss',
        'objective': 'multi:softprob',
        'num_class': len(le.classes_)
    }
    
    # Internal validation split (80% Train, 20% Val)
    X_tr, X_val, y_tr, y_val = train_test_split(
        X_train_scaled, y_train, test_size=0.2, random_state=RANDOM_STATE, stratify=y_train
    )
    
    model = xgb.XGBClassifier(**params)
    model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
    preds = model.predict(X_val)
    return f1_score(y_val, preds, average='weighted')

optuna.logging.set_verbosity(optuna.logging.WARNING)
study_xgb = optuna.create_study(direction='maximize')
study_xgb.optimize(objective_xgb, n_trials=30, show_progress_bar=True)

print("Best XGBoost Validation F1-score:", study_xgb.best_value)
print("Best XGBoost Parameters:", study_xgb.best_params)


Best trial: 12. Best value: 0.835386: 100%|██████████| 30/30 [08:44<00:00, 17.48s/it]

Best XGBoost Validation F1-score: 0.835386412990052
Best XGBoost Parameters: {'n_estimators': 492, 'max_depth': 8, 'learning_rate': 0.15460688291179658, 'subsample': 0.6631903069578137, 'colsample_bytree': 0.8920054645358594, 'gamma': 0.0005788931559471187}


### 2. LightGBM Hyperparameter Search
We tune LightGBM estimators, tree depth, number of leaves, learning rate, subsample, and column sample ratio.


In [7]:
# ============================================================
# LightGBM Hyperparameter Optimization (Optuna)
# ============================================================
def objective_lgb(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 500),
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'num_leaves': trial.suggest_int('num_leaves', 10, 150),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'random_state': RANDOM_STATE,
        'n_jobs': -1,
        'verbose': -1,
        'objective': 'multiclass',
        'num_class': len(le.classes_)
    }
    
    # Internal validation split
    X_tr, X_val, y_tr, y_val = train_test_split(
        X_train_scaled, y_train, test_size=0.2, random_state=RANDOM_STATE, stratify=y_train
    )
    
    model = lgb.LGBMClassifier(**params)
    model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)])
    preds = model.predict(X_val)
    return f1_score(y_val, preds, average='weighted')

study_lgb = optuna.create_study(direction='maximize')
study_lgb.optimize(objective_lgb, n_trials=30, show_progress_bar=True)

print("Best LightGBM Validation F1-score:", study_lgb.best_value)
print("Best LightGBM Parameters:", study_lgb.best_params)


Best trial: 11. Best value: 0.838482:  57%|█████▋    | 17/30 [04:24<03:22, 15.55s/it]


[W 2026-05-29 09:56:15,523] Trial 17 failed with parameters: {'n_estimators': 388, 'max_depth': 6, 'num_leaves': 46, 'learning_rate': 0.20334946387212494, 'subsample': 0.9281883038588572, 'colsample_bytree': 0.8589616535306179} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "c:\Users\User\AppData\Local\Programs\Python\Python312\Lib\site-packages\optuna\study\_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "C:\Users\User\AppData\Local\Temp\ipykernel_8416\3550484886.py", line 25, in objective_lgb
    model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)])
  File "c:\Users\User\AppData\Local\Programs\Python\Python312\Lib\site-packages\lightgbm\sklearn.py", line 1560, in fit
    super().fit(
  File "c:\Users\User\AppData\Local\Programs\Python\Python312\Lib\site-packages\lightgbm\sklearn.py", line 1049, in fit
    self._Booster = train(
                    ^^^^^^
  File "c:\Users\User\

KeyboardInterrupt: 

### 3. Support Vector Machine (SVM) Tuning
Since SVM training has $O(N^2 \sim N^3)$ time complexity, training on all 43,000 samples for multiple trials would take a very long time. We perform hyperparameter tuning using a stratified subset of **5,000 training samples** for speed.


In [ ]:
# ============================================================
# Support Vector Machine (SVM) Hyperparameter Optimization (Optuna)
# ============================================================
# Due to O(N^2) to O(N^3) time complexity of SVMs, hyperparameter tuning is 
# conducted on a representative stratified subset of 5,000 training samples.
def objective_svm(trial):
    params = {
        'C': trial.suggest_float('C', 0.1, 10.0, log=True),
        'gamma': trial.suggest_float('gamma', 1e-4, 1.0, log=True),
        'kernel': 'rbf',
        'random_state': RANDOM_STATE
    }
    
    X_tr, X_val, y_tr, y_val = train_test_split(
        X_train_scaled, y_train, test_size=0.2, train_size=5000, random_state=RANDOM_STATE, stratify=y_train
    )
    
    model = SVC(**params)
    model.fit(X_tr, y_tr)
    preds = model.predict(X_val)
    return f1_score(y_val, preds, average='weighted')

study_svm = optuna.create_study(direction='maximize')
study_svm.optimize(objective_svm, n_trials=15, show_progress_bar=True)

print("Best SVM Validation F1-score:", study_svm.best_value)
print("Best SVM Parameters:", study_svm.best_params)


## Model Selection & Final Training

We compare the best validation F1 scores achieved by each model, select the overall best-performing architecture, and fit it on the **entire** scaled training set.


In [ ]:
# ============================================================
# Model Selection and Final Model Training
# ============================================================
# Identify the best overall model from the Optuna validation scores
best_model_name = "XGBoost"
best_val_score = study_xgb.best_value
best_params = study_xgb.best_params

if study_lgb.best_value > best_val_score:
    best_model_name = "LightGBM"
    best_val_score = study_lgb.best_value
    best_params = study_lgb.best_params

if study_svm.best_value > best_val_score:
    best_model_name = "SVM"
    best_val_score = study_svm.best_value
    best_params = study_svm.best_params

print(f"Selected Best Model: {best_model_name} (Val F1: {best_val_score:.4f})")

# Fit the chosen model on the full training scaled dataset
if best_model_name == "XGBoost":
    final_model = xgb.XGBClassifier(**best_params, random_state=RANDOM_STATE, n_jobs=-1, eval_metric='mlogloss', objective='multi:softprob', num_class=len(le.classes_))
elif best_model_name == "LightGBM":
    final_model = lgb.LGBMClassifier(**best_params, random_state=RANDOM_STATE, n_jobs=-1, verbose=-1, objective='multiclass', num_class=len(le.classes_))
else:
    final_model = SVC(**best_params, random_state=RANDOM_STATE)

final_model.fit(X_train_scaled, y_train)
print("Final model training complete.")


## Final Evaluation on Unseen Test Set

We run predictions on the test set (which was set aside and not used during hyperparameter tuning or final fitting) and output:
1. **Classification Report**: Precision, Recall, and F1-score for each emotion class.
2. **Weighted F1-score**: Overall metric weighted by class support.
3. **Cohen's Kappa**: Measure of classification agreement compared to random chance.


In [ ]:
# ============================================================
# Final Model Evaluation on Unseen Test Dataset
# ============================================================
y_pred = final_model.predict(X_test_scaled)

print("=== Test Set Classification Report ===")
print(classification_report(y_test, y_pred, target_names=le.classes_))

f1w = f1_score(y_test, y_pred, average="weighted")
kappa = cohen_kappa_score(y_test, y_pred)
print(f"Final Test Weighted F1-Score: {f1w:.4f}")
print(f"Final Test Cohen's Kappa:      {kappa:.4f}")
